# Your First Quantum-Ready Dataset

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/tutorials/01_your_first_quantum_ready_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Ftutorials%2F01_your_first_quantum_ready_dataset.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/tutorials/01_your_first_quantum_ready_dataset.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>

You have a table of data. You want to run it through a variational quantum classifier. But quantum computers don't understand raw numbers the way classical computers do — they work with **quantum states**.

To use your data on a quantum device, you need to *encode* it: map each row of numbers to a sequence of quantum gate rotations. This process is called **quantum feature encoding**, and it has a catch — the gates expect values in specific ranges. Raw data almost never fits.

This tutorial shows you the full path from a raw dataset to encoded quantum circuits, and explains *why* each step matters.

**What you'll learn:**
- Why raw data can't go directly into a quantum circuit
- How QuPrep's `Pipeline` prepares data automatically
- How to verify your data is ready before encoding
- What an encoded circuit actually looks like

**No optional dependencies required.**

In [ ]:
# If running on Colab or Binder, install QuPrep first:
# !pip install quprep

from sklearn.datasets import load_iris

import quprep as qd

print(f"quprep {qd.__version__}")

## 1. Load a real dataset

We use the **Iris dataset** — 150 samples, 4 features (sepal and petal length/width), 3 flower species. It ships inside scikit-learn, which QuPrep already depends on, so no extra install is needed.

We take 12 samples to keep the output readable.

In [ ]:
iris = load_iris()
X_raw = iris.data[:12]   # 12 samples, 4 features each
y     = iris.target[:12]

print(f"Shape : {X_raw.shape}  (samples × features)")
print(f"Range : [{X_raw.min():.2f}, {X_raw.max():.2f}]")
print(f"Labels: {y}")

## 2. Why preprocessing matters for quantum encoding

**The core problem:** quantum gates work with angles, not arbitrary numbers.

The most common encoder — `AngleEncoder` — maps each feature to an `Ry` rotation gate. `Ry` expects an angle in **[0, π] ≈ [0, 3.14]**. Raw Iris values go up to **7.9** — more than twice that range.

What happens if you feed raw data in? The encoder silently wraps or clips values, compressing most of your data's variation into a tiny corner of the qubit's state space. Your circuit runs, but the information is garbled.

> **Classical analogy:** this is like feeding pixel values in [0, 255] into a sigmoid activation that expects [0, 1]. The network runs, but it's essentially blind to most of the variation in your images.

QuPrep's `Pipeline` solves this with three stages:

| Stage | What it does |
|---|---|
| `Imputer` | Fills missing values (NaN) with column means |
| `Scaler(minmax_pi)` | Rescales every feature to **[0, π]** |
| `AngleEncoder` | Maps each scaled value to an Ry gate rotation |

In [ ]:
dataset = qd.NumpyIngester().load(X_raw, y=y)

pipeline = qd.Pipeline(
    cleaner=qd.Imputer(strategy="mean"),
    normalizer=qd.Scaler(strategy="minmax_pi"),   # rescale to [0, π]
    encoder=qd.AngleEncoder(),
)

print("Pipeline ready: Imputer → Scaler(minmax_pi) → AngleEncoder")

## 3. Fit and transform

`fit_transform()` does two things in one call:
- **Fit**: learns the min/max of each feature from this data
- **Transform**: applies imputation, scaling, and encoding

The result is a `PipelineResult` containing the encoded circuits and per-stage metadata.

In [ ]:
import warnings

import quprep

with warnings.catch_warnings():
    warnings.simplefilter("ignore", quprep.QuPrepWarning)
    result = pipeline.fit_transform(dataset)

print(f"Scaled range : [{result.dataset.data.min():.3f}, {result.dataset.data.max():.3f}]")
print(f"Circuits     : {len(result.encoded)}")

**What just happened?** The scaler learned that sepal length goes from 4.3 to 7.9 in this batch and mapped that range to [0, π]. Every other feature got its own mapping. The encoded dataset now contains 12 circuits — one per row.

## 4. Check compatibility before trusting the output

`check_compatibility()` verifies that every value in your preprocessed dataset actually falls within the valid range for the encoder you chose. Think of it as a sanity check between preprocessing and encoding — it catches problems that would otherwise produce silently wrong circuits.

It returns a `CompatibilityReport` with `is_compatible`, `warnings`, and `errors`.

In [ ]:
report = qd.check_compatibility(qd.AngleEncoder(), result.dataset)

print(f"Compatible : {report.is_compatible}")
print(f"Warnings   : {len(report.warnings)}")
print(f"Errors     : {len(report.errors)}")

## 5. Look at a circuit

Each sample becomes a quantum circuit. The ASCII diagram shows one qubit per feature. Each qubit gets an `Ry` rotation gate whose angle is the scaled feature value.

This is the circuit you would hand to PennyLane, Qiskit, or any quantum framework.

In [ ]:
print(qd.draw_ascii(result.encoded[0]))

## 6. Export to OpenQASM 3.0

Most quantum frameworks accept **OpenQASM 3.0** as a universal circuit description language. QuPrep's `QASMExporter` converts the encoded result to a string you can paste directly into Qiskit, PennyLane, Cirq, or any QASM-compatible simulator.

For framework-specific exports (native Qiskit `QuantumCircuit`, PennyLane tapes, etc.), see [how-to/export_to_frameworks](../how-to/export_to_frameworks.ipynb).

In [ ]:
qasm = qd.QASMExporter().export(result.encoded[0])
print(qasm)

## Next steps

You've gone from a raw dataset to encoded quantum circuits in six steps. Here's where to go next:

| I want to... | Go to |
|---|---|
| Handle messy data (NaN, outliers, class imbalance) | [tutorials/02_real_world_messy_data](02_real_world_messy_data.ipynb) |
| Connect to Qiskit with an auto-generated pipeline | [tutorials/03_end_to_end_with_a_framework](03_end_to_end_with_a_framework.ipynb) |
| Pick the best encoder for my task | [how-to/choose_an_encoder](../how-to/choose_an_encoder.ipynb) |
| Export to PennyLane, Cirq, TKET, Braket | [how-to/export_to_frameworks](../how-to/export_to_frameworks.ipynb) |
| Understand what each stage did to my data | [how-to/validate_before_encoding](../how-to/validate_before_encoding.ipynb) |